# Chess ELO-difference regression with Stockfish move-quality features

This notebook adapts the Stockfish-feature ELO regression workflow to predict
**rating difference** rather than absolute rating.

The target is player-centric:

```text
rating_diff = player_elo - opponent_elo
```

Each analyzed game is converted into two rows, one from White's perspective and
one from Black's perspective. The two rows have opposite targets. To avoid
leakage, the train/validation/test split is done by `game_index`, so the two
perspectives from the same game always stay in the same split.

The main idea is:

1. Stream a small/subsampled Lichess PGN file.
2. Filter bad or misleading games, including provisional/new-player 1500 cases.
3. Use Stockfish to estimate centipawn loss and best-move agreement.
4. Convert each game into player-centric rows with own/opponent move-quality
   summaries.
5. Train XGBoost to predict `player_elo - opponent_elo`.
6. Compare against simple baselines and inspect residuals.


## Notes on dependencies

Python packages used here:

```bash
pip install python-chess zstandard pandas pyarrow tqdm scikit-learn xgboost matplotlib seaborn joblib
```

You also need a Stockfish binary. On many Linux/Colab setups:

```bash
sudo apt-get update
sudo apt-get install -y stockfish
```

If `stockfish` is not on your PATH, set `STOCKFISH_PATH` below manually.

In [ ]:
from pathlib import Path
import io
import os
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
import urllib.request
import zstandard as zstd
import chess
import chess.pgn
import chess.engine

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
  mean_absolute_error,
  mean_squared_error,
  median_absolute_error,
  r2_score,
)

from xgboost import XGBRegressor
import joblib

warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

## Configuration

The default is a small subset of the April 2014 standard rated Lichess data,
because that is the dataset we have already been using. You can switch to an
earlier month if you want a physically smaller download, but the more important
control is `MAX_GAMES_TO_READ` and `N_GAMES_ENGINE`.

In [ ]:
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = DATA_DIR / "models"

for path in [RAW_DIR, PROCESSED_DIR, MODEL_DIR]:
  path.mkdir(parents=True, exist_ok=True)

# Lichess monthly database file.
year = 2014
month = 4

filename = f"lichess_db_standard_rated_{year}-{month:02d}.pgn.zst"
url = f"https://database.lichess.org/standard/{filename}"
zst_path = RAW_DIR / filename

# Keep this modest. PGN parsing is cheap; Stockfish analysis is not.
MAX_GAMES_TO_READ = 36_000
N_GAMES_ENGINE = 1_000

# Analyze only the first N plies of each game. 30 plies = 15 full moves.
# Increasing this improves signal but scales linearly in runtime.
ENGINE_MAX_PLIES = 30
ENGINE_DEPTH = 8
ENGINE_THREADS = 1
ENGINE_HASH_MB = 128

# Cap extreme mate-conversion losses so one forced mate does not dominate.
CP_LOSS_CLIP = 2_000
MATE_SCORE = 100_000

# Change this manually if shutil.which("stockfish") does not find your binary.
# On macOS with Homebrew this is often /opt/homebrew/bin/stockfish.
STOCKFISH_PATH = os.environ.get("STOCKFISH_PATH") or shutil.which("stockfish")

print("PGN URL:", url)
print("Local path:", zst_path)
print("Stockfish path:", STOCKFISH_PATH)

## Download PGN file

This cell only downloads the file if it is not already present locally.

In [ ]:
if not zst_path.exists():
  print(f"Downloading {filename}...")
  urllib.request.urlretrieve(url, zst_path)
  print("Done.")
else:
  print("File already exists.")

## PGN streaming helpers

The parser streams directly from the `.zst` file, so we do not have to decompress
the whole PGN file to disk.

In [ ]:
def iter_pgn_games_from_zst(zst_path, max_games=None):
  """Yield python-chess Game objects from a compressed Lichess PGN file."""
  dctx = zstd.ZstdDecompressor()

  with open(zst_path, "rb") as compressed:
    with dctx.stream_reader(compressed) as reader:
      text_stream = io.TextIOWrapper(reader, encoding="utf-8")

      n_games = 0
      while True:
        game = chess.pgn.read_game(text_stream)

        if game is None:
          break

        yield game
        n_games += 1

        if max_games is not None and n_games >= max_games:
          break


def parse_rating(value):
  """Parse Lichess rating headers, keeping provisional '?' information separate."""
  if value is None:
    return np.nan, False

  text = str(value).strip()
  is_provisional = text.endswith("?")
  text = text.replace("?", "")

  try:
    return float(text), is_provisional
  except ValueError:
    return np.nan, is_provisional


def game_to_light_record(game):
  """Extract metadata plus SAN/UCI moves without doing engine analysis yet."""
  headers = game.headers
  board = game.board()

  moves_san = []
  moves_uci = []

  for move in game.mainline_moves():
    moves_san.append(board.san(move))
    moves_uci.append(move.uci())
    board.push(move)

  white_elo, white_provisional = parse_rating(headers.get("WhiteElo"))
  black_elo, black_provisional = parse_rating(headers.get("BlackElo"))

  return {
    "event": headers.get("Event"),
    "site": headers.get("Site"),
    "date": headers.get("UTCDate", headers.get("Date")),
    "time": headers.get("UTCTime"),
    "white": headers.get("White"),
    "black": headers.get("Black"),
    "result": headers.get("Result"),
    "white_elo": white_elo,
    "black_elo": black_elo,
    "white_provisional": white_provisional,
    "black_provisional": black_provisional,
    "white_rating_diff": headers.get("WhiteRatingDiff"),
    "black_rating_diff": headers.get("BlackRatingDiff"),
    "eco": headers.get("ECO"),
    "opening": headers.get("Opening"),
    "time_control": headers.get("TimeControl"),
    "termination": headers.get("Termination"),
    "num_plies": len(moves_uci),
    "num_full_moves": len(moves_uci) / 2,
    "moves_san": " ".join(moves_san),
    "moves_uci": " ".join(moves_uci),
  }

## Parse a small PGN subset

This is the first quality-of-life improvement: cache the parsed metadata so that
changing the model does not require re-reading PGNs every time.

In [ ]:
metadata_path = PROCESSED_DIR / (
  f"lichess_{year}_{month:02d}_metadata_first_{MAX_GAMES_TO_READ}.parquet"
)

if metadata_path.exists():
  df_raw = pd.read_parquet(metadata_path)
  print(f"Loaded cached metadata: {metadata_path}")
else:
  records = []
  game_iter = iter_pgn_games_from_zst(
    zst_path,
    max_games=MAX_GAMES_TO_READ,
  )

  for game in tqdm(game_iter, total=MAX_GAMES_TO_READ):
    records.append(game_to_light_record(game))

  df_raw = pd.DataFrame(records)
  df_raw.to_parquet(metadata_path, index=False)
  print(f"Wrote: {metadata_path}")

df_raw.head()

## Basic data exploration

We keep the exploratory part because this problem is very sensitive to rating
spikes, short games, time controls, and result imbalance.

In [ ]:
print(df_raw.shape)
df_raw.info()

In [ ]:
inspect_cols = [
  "white_elo",
  "black_elo",
  "result",
  "eco",
  "opening",
  "time_control",
  "termination",
  "num_plies",
  "white_provisional",
  "black_provisional",
]

df_raw[inspect_cols].head(20)

In [ ]:
df_raw[["white_elo", "black_elo", "num_plies"]].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df_raw["white_elo"].hist(bins=60, alpha=0.6, ax=ax, label="White")
df_raw["black_elo"].hist(bins=60, alpha=0.6, ax=ax, label="Black")
ax.set_xlabel("Rating")
ax.set_ylabel("Games")
ax.set_title("Raw rating distributions")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df_raw["num_plies"].hist(bins=80, ax=ax)
ax.set_xlabel("Number of plies")
ax.set_ylabel("Games")
ax.set_title("Game length distribution")
plt.tight_layout()
plt.show()

In [ ]:
print("Results:")
display(df_raw["result"].value_counts(dropna=False).to_frame("count"))

print("\nTermination:")
display(df_raw["termination"].value_counts(dropna=False).head(20).to_frame("count"))

print("\nTime controls:")
display(df_raw["time_control"].value_counts(dropna=False).head(20).to_frame("count"))

print("\nOpenings:")
display(df_raw["opening"].value_counts(dropna=False).head(20).to_frame("count"))

## Filter obvious data issues

The exact 1500 spike is suspicious in old Lichess data because new/provisional
accounts start there. We remove exact 1500 ratings by default and also remove
ratings explicitly marked as provisional with `?` when present.

In [ ]:
MIN_PLIES = 20
REMOVE_EXACT_1500 = True
REMOVE_PROVISIONAL = True

valid_mask = (
  df_raw["white_elo"].notna()
  & df_raw["black_elo"].notna()
  & df_raw["result"].isin(["1-0", "0-1", "1/2-1/2"])
  & df_raw["moves_uci"].notna()
  & df_raw["num_plies"].ge(MIN_PLIES)
)

if REMOVE_EXACT_1500:
  valid_mask &= ~(
    df_raw["white_elo"].eq(1500)
    | df_raw["black_elo"].eq(1500)
  )

if REMOVE_PROVISIONAL:
  valid_mask &= ~(
    df_raw["white_provisional"].fillna(False)
    | df_raw["black_provisional"].fillna(False)
  )

df_games = df_raw.loc[valid_mask].copy()

print("Rows before filtering:", len(df_raw))
print("Rows after filtering: ", len(df_games))
print("Removed:              ", len(df_raw) - len(df_games))

df_games[["white_elo", "black_elo", "num_plies"]].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df_games["white_elo"].hist(bins=60, alpha=0.6, ax=ax, label="White")
df_games["black_elo"].hist(bins=60, alpha=0.6, ax=ax, label="Black")
ax.axvline(1500, linestyle="--", label="1500")
ax.set_xlabel("Rating")
ax.set_ylabel("Games")
ax.set_title("Filtered rating distributions")
ax.legend()
plt.tight_layout()
plt.show()

## Draw a small engine-analysis sample

Engine analysis is the bottleneck. To keep class balance reasonable, we sample
from broad average-rating bins rather than simply taking the first `N` games.

In [ ]:
def add_average_rating_bin(df):
  df = df.copy()

  df["white_elo"] = pd.to_numeric(df["white_elo"], errors="coerce")
  df["black_elo"] = pd.to_numeric(df["black_elo"], errors="coerce")
  df["avg_elo"] = (df["white_elo"] + df["black_elo"]) / 2

  df["avg_elo_bin"] = pd.cut(
    df["avg_elo"],
    bins=[0, 1200, 1500, 1800, 2100, 3000],
    labels=["<1200", "1200-1499", "1500-1799", "1800-2099", "2100+"],
    include_lowest=True,
    right=False,
  )

  return df


df_games = add_average_rating_bin(df_games)

print("Full parsed game table:")
print(df_games.shape)
display(
  df_games["avg_elo_bin"]
  .value_counts(dropna=False)
  .sort_index()
  .to_frame("games")
)

# Keep only games with valid ratings and valid average-rating bin.
df_games_with_elo = (
  df_games
  .dropna(subset=["white_elo", "black_elo", "avg_elo", "avg_elo_bin"])
  .copy()
  .reset_index(drop=True)
)

print("After ELO filtering:")
print(df_games_with_elo.shape)
display(
  df_games_with_elo["avg_elo_bin"]
  .value_counts(dropna=False)
  .sort_index()
  .to_frame("games")
)

sample_size = min(N_GAMES_ENGINE, len(df_games_with_elo))

if sample_size == 0:
  raise ValueError("No valid games remain after ELO filtering.")

# Stratified sample by average ELO bin. This avoids groupby/apply index
# surprises and samples only from the filtered dataframe.
sampled_parts = []

for bin_name, g in df_games_with_elo.groupby("avg_elo_bin", observed=True):
  n_bin = round(sample_size * len(g) / len(df_games_with_elo))
  n_bin = max(1, n_bin)
  n_bin = min(n_bin, len(g))

  sampled_parts.append(
    g.sample(n=n_bin, random_state=RANDOM_STATE)
  )

df_engine_input = pd.concat(sampled_parts, axis=0)

# Trim or top up from the filtered dataframe only.
if len(df_engine_input) > sample_size:
  df_engine_input = df_engine_input.sample(
    n=sample_size,
    random_state=RANDOM_STATE,
  )

elif len(df_engine_input) < sample_size:
  missing = sample_size - len(df_engine_input)
  remaining_pool = df_games_with_elo.drop(df_engine_input.index)

  extra = remaining_pool.sample(
    n=min(missing, len(remaining_pool)),
    random_state=RANDOM_STATE,
  )

  df_engine_input = pd.concat([df_engine_input, extra], axis=0)

df_engine_input = (
  df_engine_input
  .sample(frac=1, random_state=RANDOM_STATE)
  .reset_index(drop=True)
  .copy()
)

# Hard checks so we fail here, not many cells later.
assert len(df_engine_input) == sample_size
assert df_engine_input["white_elo"].notna().all()
assert df_engine_input["black_elo"].notna().all()
assert df_engine_input["avg_elo"].notna().all()
assert df_engine_input["avg_elo_bin"].notna().all()

print("Final engine input:")
print(df_engine_input.shape)
display(
  df_engine_input["avg_elo_bin"]
  .value_counts(dropna=False)
  .sort_index()
  .to_frame("games")
)

## Stockfish feature extraction

For each move, we compare the engine's best-line evaluation before the move to
the evaluation after the actual played move. From the moving player's
perspective:

`centipawn_loss = best_position_eval_before_move - eval_after_played_move`

This is clipped to avoid one mate sequence dominating all summary statistics.
The output is one row per game with separate White/Black engine summaries.

In [ ]:
def require_stockfish_path(stockfish_path):
  if stockfish_path is None:
    raise FileNotFoundError(
      "Stockfish was not found. Install it or set STOCKFISH_PATH manually."
    )

  path = Path(stockfish_path)
  if not path.exists() and shutil.which(str(stockfish_path)) is None:
    raise FileNotFoundError(
      f"Stockfish path does not exist: {stockfish_path}"
    )


def score_to_cp(score, pov_color):
  """Convert a python-chess PovScore/Score to centipawns from pov_color."""
  return score.pov(pov_color).score(mate_score=MATE_SCORE)


def summarize_cp_losses(losses, prefix):
  losses = np.asarray(losses, dtype=float)

  if len(losses) == 0:
    return {
      f"{prefix}_moves_analyzed": 0,
      f"{prefix}_acpl": np.nan,
      f"{prefix}_median_cpl": np.nan,
      f"{prefix}_std_cpl": np.nan,
      f"{prefix}_p75_cpl": np.nan,
      f"{prefix}_p90_cpl": np.nan,
      f"{prefix}_max_cpl": np.nan,
      f"{prefix}_inaccuracy_rate": np.nan,
      f"{prefix}_mistake_rate": np.nan,
      f"{prefix}_blunder_rate": np.nan,
    }

  return {
    f"{prefix}_moves_analyzed": len(losses),
    f"{prefix}_acpl": float(np.mean(losses)),
    f"{prefix}_median_cpl": float(np.median(losses)),
    f"{prefix}_std_cpl": float(np.std(losses)),
    f"{prefix}_p75_cpl": float(np.percentile(losses, 75)),
    f"{prefix}_p90_cpl": float(np.percentile(losses, 90)),
    f"{prefix}_max_cpl": float(np.max(losses)),
    f"{prefix}_inaccuracy_rate": float(np.mean(losses >= 50)),
    f"{prefix}_mistake_rate": float(np.mean(losses >= 100)),
    f"{prefix}_blunder_rate": float(np.mean(losses >= 300)),
  }


def phase_summaries(losses_by_ply, prefix):
  """Summarize opening/middlegame/endgame losses by absolute ply index."""
  out = {}
  phases = {
    "opening": lambda ply: ply <= 10,
    "middlegame": lambda ply: 10 < ply <= 30,
    "late": lambda ply: ply > 30,
  }

  for phase, selector in phases.items():
    phase_losses = [loss for ply, loss in losses_by_ply if selector(ply)]
    if len(phase_losses) == 0:
      out[f"{prefix}_{phase}_acpl"] = np.nan
    else:
      out[f"{prefix}_{phase}_acpl"] = float(np.mean(phase_losses))

  return out


def result_for_color(result, color):
  if result == "1-0":
    return 1.0 if color == chess.WHITE else 0.0
  if result == "0-1":
    return 0.0 if color == chess.WHITE else 1.0
  if result == "1/2-1/2":
    return 0.5
  return np.nan

In [ ]:
def analyze_game_with_engine(row, engine, max_plies=30, depth=8):
  """Analyze one game row and return game-level engine features."""
  board = chess.Board()
  moves = []

  for uci in str(row["moves_uci"]).split():
    try:
      move = chess.Move.from_uci(uci)
    except ValueError:
      break

    if move not in board.legal_moves:
      break

    moves.append(move)
    board.push(move)

  board.reset()

  losses = {
    chess.WHITE: [],
    chess.BLACK: [],
  }
  losses_by_ply = {
    chess.WHITE: [],
    chess.BLACK: [],
  }
  best_matches = {
    chess.WHITE: [],
    chess.BLACK: [],
  }
  eval_before_values = {
    chess.WHITE: [],
    chess.BLACK: [],
  }

  limit = chess.engine.Limit(depth=depth)

  for ply_idx, move in enumerate(moves[:max_plies], start=1):
    mover = board.turn

    info_before = engine.analyse(board, limit)
    best_score_cp = score_to_cp(info_before["score"], mover)
    best_move = None
    if "pv" in info_before and len(info_before["pv"]) > 0:
      best_move = info_before["pv"][0]

    board.push(move)

    info_after = engine.analyse(board, limit)
    played_score_cp = score_to_cp(info_after["score"], mover)

    cp_loss = best_score_cp - played_score_cp
    cp_loss = max(0, cp_loss)
    cp_loss = min(cp_loss, CP_LOSS_CLIP)

    losses[mover].append(cp_loss)
    losses_by_ply[mover].append((ply_idx, cp_loss))
    eval_before_values[mover].append(best_score_cp)
    best_matches[mover].append(float(best_move == move) if best_move else np.nan)

  out = {
    "game_index": row.name,
    "white_elo": row["white_elo"],
    "black_elo": row["black_elo"],
    "result": row["result"],
    "eco": row["eco"],
    "opening": row["opening"],
    "time_control": row["time_control"],
    "termination": row["termination"],
    "num_plies": row["num_plies"],
    "num_full_moves": row["num_full_moves"],
    "engine_plies_analyzed": min(len(moves), max_plies),
  }

  for color, prefix in [(chess.WHITE, "white"), (chess.BLACK, "black")]:
    out.update(summarize_cp_losses(losses[color], prefix))
    out.update(phase_summaries(losses_by_ply[color], prefix))

    matches = np.asarray(best_matches[color], dtype=float)
    evals = np.asarray(eval_before_values[color], dtype=float)

    out[f"{prefix}_best_move_rate"] = (
      float(np.nanmean(matches)) if len(matches) else np.nan
    )
    out[f"{prefix}_mean_eval_before"] = (
      float(np.nanmean(evals)) if len(evals) else np.nan
    )
    out[f"{prefix}_result_score"] = result_for_color(row["result"], color)

  out["acpl_gap_white_minus_black"] = out["white_acpl"] - out["black_acpl"]
  out["best_move_rate_gap_white_minus_black"] = (
    out["white_best_move_rate"] - out["black_best_move_rate"]
  )

  return out

## Run or load cached Stockfish analysis

This cell can take several minutes depending on `N_GAMES_ENGINE`,
`ENGINE_MAX_PLIES`, and `ENGINE_DEPTH`. The result is cached as Parquet.

In [ ]:
engine_features_path = PROCESSED_DIR / (
  f"lichess_{year}_{month:02d}_stockfish_features_"
  f"games{len(df_engine_input)}_plies{ENGINE_MAX_PLIES}_depth{ENGINE_DEPTH}.parquet"
)

if engine_features_path.exists():
  df_engine = pd.read_parquet(engine_features_path)
  print(f"Loaded cached engine features: {engine_features_path}")
else:
  require_stockfish_path(STOCKFISH_PATH)

  engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)
  engine.configure({
    "Threads": ENGINE_THREADS,
    "Hash": ENGINE_HASH_MB,
  })

  feature_records = []
  try:
    for _, row in tqdm(
      df_engine_input.iterrows(),
      total=len(df_engine_input),
      desc="Stockfish analysis",
    ):
      try:
        feature_records.append(
          analyze_game_with_engine(
            row,
            engine=engine,
            max_plies=ENGINE_MAX_PLIES,
            depth=ENGINE_DEPTH,
          )
        )
      except Exception as exc:
        print(f"Skipping game index {row.name}: {exc}")
  finally:
    engine.quit()

  df_engine = pd.DataFrame(feature_records)
  df_engine.to_parquet(engine_features_path, index=False)
  print(f"Wrote: {engine_features_path}")

df_engine.head()

## Explore engine-derived features

Now we can check whether the engine features have sensible distributions and
whether they visibly separate rating groups.

In [ ]:
df_engine.shape

In [ ]:
engine_summary_cols = [
  "white_acpl",
  "black_acpl",
  "white_best_move_rate",
  "black_best_move_rate",
  "white_blunder_rate",
  "black_blunder_rate",
  "acpl_gap_white_minus_black",
  "engine_plies_analyzed",
]

df_engine[engine_summary_cols].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df_engine["white_acpl"].hist(bins=50, alpha=0.6, ax=ax, label="White")
df_engine["black_acpl"].hist(bins=50, alpha=0.6, ax=ax, label="Black")
ax.set_xlabel("Average centipawn loss")
ax.set_ylabel("Games")
ax.set_title("Engine-estimated move quality")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
plot_df = df_engine.copy()
plot_df["avg_elo"] = (plot_df["white_elo"] + plot_df["black_elo"]) / 2
plot_df["avg_acpl"] = (plot_df["white_acpl"] + plot_df["black_acpl"]) / 2

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(plot_df["avg_elo"], plot_df["avg_acpl"], alpha=0.35, s=12)
ax.set_xlabel("Average game rating")
ax.set_ylabel("Average game ACPL")
ax.set_title("Rating vs Stockfish move quality")
plt.tight_layout()
plt.show()

In [ ]:
# Correlations give a quick sanity check. We expect better players to have
# lower ACPL and fewer blunders, though the relationship will be noisy.
corr_cols = [
  "white_elo",
  "black_elo",
  "white_acpl",
  "black_acpl",
  "white_best_move_rate",
  "black_best_move_rate",
  "white_blunder_rate",
  "black_blunder_rate",
  "num_plies",
]

df_engine[corr_cols].corr(numeric_only=True).round(3)

## Convert games into player-centric ELO-difference rows

In [ ]:
RATING_BINS = [0, 1200, 1500, 1800, 2100, 3000]
RATING_LABELS = ["<1200", "1200-1499", "1500-1799", "1800-2099", "2100+"]

DIFF_BINS = [-2000, -600, -300, -100, 100, 300, 600, 2000]
DIFF_LABELS = [
  "< -600",
  "-600 to -300",
  "-300 to -100",
  "-100 to 100",
  "100 to 300",
  "300 to 600",
  "> 600",
]

own_suffixes = [
  "moves_analyzed",
  "acpl",
  "median_cpl",
  "std_cpl",
  "p75_cpl",
  "p90_cpl",
  "max_cpl",
  "inaccuracy_rate",
  "mistake_rate",
  "blunder_rate",
  "opening_acpl",
  "middlegame_acpl",
  "late_acpl",
  "best_move_rate",
  "mean_eval_before",
  "result_score",
]


def make_player_rows(df):
  rows = []

  for _, row in df.iterrows():
    game_index = row.get("game_index", row.name)

    for color_name, opp_name, is_white in [
      ("white", "black", 1),
      ("black", "white", 0),
    ]:
      player_rating = row[f"{color_name}_elo"]
      opponent_rating = row[f"{opp_name}_elo"]
      rating_diff = player_rating - opponent_rating

      out = {
        "game_index": game_index,
        "rating": player_rating,
        "opponent_rating": opponent_rating,
        "rating_diff": rating_diff,
        "abs_rating_diff": abs(rating_diff),
        "rating_bin": pd.cut(
          [player_rating],
          bins=RATING_BINS,
          labels=RATING_LABELS,
          include_lowest=True,
          right=False,
        )[0],
        "opponent_rating_bin": pd.cut(
          [opponent_rating],
          bins=RATING_BINS,
          labels=RATING_LABELS,
          include_lowest=True,
          right=False,
        )[0],
        "rating_diff_bin": pd.cut(
          [rating_diff],
          bins=DIFF_BINS,
          labels=DIFF_LABELS,
          include_lowest=True,
          right=False,
        )[0],
        "is_white": is_white,
        "result": row["result"],
        "eco": row["eco"],
        "time_control": row["time_control"],
        "termination": row["termination"],
        "num_plies": row["num_plies"],
        "num_full_moves": row["num_full_moves"],
        "engine_plies_analyzed": row["engine_plies_analyzed"],
      }

      for suffix in own_suffixes:
        out[f"own_{suffix}"] = row[f"{color_name}_{suffix}"]
        out[f"opp_{suffix}"] = row[f"{opp_name}_{suffix}"]

      out["own_minus_opp_acpl"] = out["own_acpl"] - out["opp_acpl"]
      out["own_minus_opp_best_move_rate"] = (
        out["own_best_move_rate"] - out["opp_best_move_rate"]
      )
      out["own_minus_opp_blunder_rate"] = (
        out["own_blunder_rate"] - out["opp_blunder_rate"]
      )
      out["own_minus_opp_mistake_rate"] = (
        out["own_mistake_rate"] - out["opp_mistake_rate"]
      )
      out["own_minus_opp_inaccuracy_rate"] = (
        out["own_inaccuracy_rate"] - out["opp_inaccuracy_rate"]
      )
      out["own_minus_opp_opening_acpl"] = (
        out["own_opening_acpl"] - out["opp_opening_acpl"]
      )
      out["own_minus_opp_middlegame_acpl"] = (
        out["own_middlegame_acpl"] - out["opp_middlegame_acpl"]
      )
      out["own_minus_opp_late_acpl"] = (
        out["own_late_acpl"] - out["opp_late_acpl"]
      )

      rows.append(out)

  return pd.DataFrame(rows)


df_player = make_player_rows(df_engine)

df_player.head()

In [ ]:
print(df_player.shape)
display(df_player["rating_bin"].value_counts(dropna=False).sort_index().to_frame("players"))
display(df_player["rating_diff_bin"].value_counts(dropna=False).sort_index().to_frame("players"))
display(
  df_player[
    [
      "rating",
      "opponent_rating",
      "rating_diff",
      "own_acpl",
      "opp_acpl",
      "own_minus_opp_acpl",
      "own_best_move_rate",
      "opp_best_move_rate",
    ]
  ].describe()
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for label, group in df_player.groupby("rating_diff_bin", observed=True):
  if len(group) >= 5:
    group["own_minus_opp_acpl"].plot(kind="kde", ax=ax, label=str(label))

ax.axvline(0, linestyle="--")
ax.set_xlabel("Own ACPL - opponent ACPL")
ax.set_title("Relative move quality by ELO-difference bin")
ax.legend(title="ELO diff bin")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
  df_player["rating_diff"],
  df_player["own_minus_opp_acpl"],
  alpha=0.35,
  s=16,
)
ax.axhline(0, linestyle="--")
ax.axvline(0, linestyle="--")
ax.set_xlabel("True ELO difference: player - opponent")
ax.set_ylabel("Own ACPL - opponent ACPL")
ax.set_title("Relative move quality vs rating difference")
plt.tight_layout()
plt.show()

## Select features and ELO-difference target

In [ ]:
target_col = "rating_diff"

numeric_features = [
  "is_white",
  "num_plies",
  "num_full_moves",
  "engine_plies_analyzed",
  "own_moves_analyzed",
  "opp_moves_analyzed",
  "own_acpl",
  "opp_acpl",
  "own_median_cpl",
  "opp_median_cpl",
  "own_std_cpl",
  "opp_std_cpl",
  "own_p75_cpl",
  "opp_p75_cpl",
  "own_p90_cpl",
  "opp_p90_cpl",
  "own_max_cpl",
  "opp_max_cpl",
  "own_inaccuracy_rate",
  "opp_inaccuracy_rate",
  "own_mistake_rate",
  "opp_mistake_rate",
  "own_blunder_rate",
  "opp_blunder_rate",
  "own_opening_acpl",
  "opp_opening_acpl",
  "own_middlegame_acpl",
  "opp_middlegame_acpl",
  "own_late_acpl",
  "opp_late_acpl",
  "own_best_move_rate",
  "opp_best_move_rate",
  "own_mean_eval_before",
  "opp_mean_eval_before",
  "own_result_score",
  "opp_result_score",
  "own_minus_opp_acpl",
  "own_minus_opp_best_move_rate",
  "own_minus_opp_blunder_rate",
  "own_minus_opp_mistake_rate",
  "own_minus_opp_inaccuracy_rate",
  "own_minus_opp_opening_acpl",
  "own_minus_opp_middlegame_acpl",
  "own_minus_opp_late_acpl",
]

categorical_features = [
  "result",
  "eco",
  "time_control",
  "termination",
]

model_df = df_player.dropna(
  subset=[
    "game_index",
    target_col,
    "rating",
    "opponent_rating",
    "rating_bin",
    "opponent_rating_bin",
    "rating_diff_bin",
  ]
).copy()

model_df[target_col] = pd.to_numeric(model_df[target_col], errors="coerce")
model_df = model_df.dropna(subset=[target_col]).copy()

print("Player rows available for ELO-difference regression:")
print(model_df.shape)
display(model_df["rating_diff"].describe().to_frame("rating_diff"))
display(model_df["rating_diff_bin"].value_counts(dropna=False).sort_index().to_frame("players"))

In [ ]:
X_raw = model_df[numeric_features + categorical_features].copy()
y = model_df[target_col].astype(float).copy()

# Keep these only for splitting and diagnostics.
groups = model_df["game_index"].copy()
y_diff_bins = model_df["rating_diff_bin"].astype(str).copy()
own_rating_bins = model_df["rating_bin"].astype(str).copy()

# Simple imputation. Tree models tolerate rough imputations well, but they do
# not accept NaNs in every configuration/version.
for col in numeric_features:
  X_raw[col] = X_raw[col].replace([np.inf, -np.inf], np.nan)
  X_raw[col] = X_raw[col].fillna(X_raw[col].median())

for col in categorical_features:
  X_raw[col] = X_raw[col].fillna("MISSING").astype(str)

X = pd.get_dummies(
  X_raw,
  columns=categorical_features,
  dummy_na=False,
)

X.columns = (
  X.columns
  .astype(str)
  .str.replace("[", "(", regex=False)
  .str.replace("]", ")", regex=False)
  .str.replace("<", "lt", regex=False)
  .str.replace(">", "gt", regex=False)
)

print("Examples:", X.shape[0])
print("Features:", X.shape[1])
display(y.describe().to_frame("rating_diff"))
X.head()

## Train / validation / test split by game

In [ ]:
# Split by game_index, not by player-row. This prevents the White row from a
# game landing in train while the Black row from the same game lands in valid/test.
unique_games = (
  model_df[["game_index"]]
  .drop_duplicates()
  .sample(frac=1, random_state=RANDOM_STATE)
  ["game_index"]
  .to_numpy()
)

train_game_ids, rem_game_ids = train_test_split(
  unique_games,
  train_size=0.70,
  random_state=RANDOM_STATE,
)

valid_game_ids, test_game_ids = train_test_split(
  rem_game_ids,
  train_size=0.50,
  random_state=RANDOM_STATE,
)

train_mask = model_df["game_index"].isin(train_game_ids)
valid_mask = model_df["game_index"].isin(valid_game_ids)
test_mask = model_df["game_index"].isin(test_game_ids)

X_train = X.loc[train_mask].copy()
X_valid = X.loc[valid_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_valid = y.loc[valid_mask].copy()
y_test = y.loc[test_mask].copy()

diff_bins_train = y_diff_bins.loc[train_mask].copy()
diff_bins_valid = y_diff_bins.loc[valid_mask].copy()
diff_bins_test = y_diff_bins.loc[test_mask].copy()

own_bins_train = own_rating_bins.loc[train_mask].copy()
own_bins_valid = own_rating_bins.loc[valid_mask].copy()
own_bins_test = own_rating_bins.loc[test_mask].copy()

print("Train:", X_train.shape, "games:", len(train_game_ids))
display(diff_bins_train.value_counts().sort_index().to_frame("players"))

print("Valid:", X_valid.shape, "games:", len(valid_game_ids))
display(diff_bins_valid.value_counts().sort_index().to_frame("players"))

print("Test:", X_test.shape, "games:", len(test_game_ids))
display(diff_bins_test.value_counts().sort_index().to_frame("players"))

# Leakage check: every game should appear in exactly one split.
assert set(train_game_ids).isdisjoint(set(valid_game_ids))
assert set(train_game_ids).isdisjoint(set(test_game_ids))
assert set(valid_game_ids).isdisjoint(set(test_game_ids))

## Baselines

In [ ]:
def regression_metrics(y_true, y_pred):
  y_true = np.asarray(y_true, dtype=float)
  y_pred = np.asarray(y_pred, dtype=float)

  return {
    "MAE": mean_absolute_error(y_true, y_pred),
    "MedianAE": median_absolute_error(y_true, y_pred),
    "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
    "R2": r2_score(y_true, y_pred),
    "Bias_mean_pred_minus_true": np.mean(y_pred - y_true),
  }


def evaluate_regressor(y_true, y_pred, label):
  metrics = regression_metrics(y_true, y_pred)

  print(label)
  print("-" * len(label))
  for name, value in metrics.items():
    print(f"{name:25s}: {value:10.3f}")

  return metrics


def plot_regression_diagnostics(y_true, y_pred, title_prefix):
  y_true = pd.Series(y_true).reset_index(drop=True)
  y_pred = pd.Series(y_pred).reset_index(drop=True)
  residual = y_pred - y_true

  fig, ax = plt.subplots(figsize=(6, 6))
  ax.scatter(y_true, y_pred, alpha=0.35, s=18)
  lo = min(y_true.min(), y_pred.min())
  hi = max(y_true.max(), y_pred.max())
  ax.plot([lo, hi], [lo, hi], linestyle="--")
  ax.axhline(0, linestyle=":")
  ax.axvline(0, linestyle=":")
  ax.set_xlabel("True ELO difference")
  ax.set_ylabel("Predicted ELO difference")
  ax.set_title(f"{title_prefix}: predicted vs true")
  plt.tight_layout()
  plt.show()

  fig, ax = plt.subplots(figsize=(8, 4))
  residual.hist(bins=50, ax=ax)
  ax.axvline(0, linestyle="--")
  ax.set_xlabel("Prediction residual: predicted - true ELO difference")
  ax.set_ylabel("Players")
  ax.set_title(f"{title_prefix}: residual distribution")
  plt.tight_layout()
  plt.show()

  fig, ax = plt.subplots(figsize=(8, 4))
  ax.scatter(y_true, residual, alpha=0.35, s=18)
  ax.axhline(0, linestyle="--")
  ax.axvline(0, linestyle=":")
  ax.set_xlabel("True ELO difference")
  ax.set_ylabel("Residual")
  ax.set_title(f"{title_prefix}: residuals vs true ELO difference")
  plt.tight_layout()
  plt.show()

In [ ]:
mean_diff = float(y_train.mean())

valid_zero_pred = np.zeros(shape=len(y_valid))
test_zero_pred = np.zeros(shape=len(y_test))

valid_mean_pred = np.full(shape=len(y_valid), fill_value=mean_diff)
test_mean_pred = np.full(shape=len(y_test), fill_value=mean_diff)

baseline_valid_zero_metrics = evaluate_regressor(
  y_valid,
  valid_zero_pred,
  "Validation zero-difference baseline",
)

baseline_test_zero_metrics = evaluate_regressor(
  y_test,
  test_zero_pred,
  "Test zero-difference baseline",
)

baseline_valid_mean_metrics = evaluate_regressor(
  y_valid,
  valid_mean_pred,
  "Validation mean-difference baseline",
)

baseline_test_mean_metrics = evaluate_regressor(
  y_test,
  test_mean_pred,
  "Test mean-difference baseline",
)

## XGBoost ELO-difference regressor

In [ ]:
xgb_reg = XGBRegressor(
  objective="reg:squarederror",
  eval_metric="rmse",
  n_estimators=500,
  max_depth=3,
  learning_rate=0.035,
  subsample=0.85,
  colsample_bytree=0.85,
  min_child_weight=3,
  reg_lambda=3.0,
  reg_alpha=0.0,
  random_state=RANDOM_STATE,
  n_jobs=-1,
)

xgb_reg.fit(
  X_train,
  y_train,
  eval_set=[(X_train, y_train), (X_valid, y_valid)],
  verbose=False,
)

In [ ]:
valid_pred = xgb_reg.predict(X_valid)
test_pred = xgb_reg.predict(X_test)

# ELO differences outside this range are not useful for this analysis. This
# rarely matters, but it makes residual plots more stable.
valid_pred = np.clip(valid_pred, -1600, 1600)
test_pred = np.clip(test_pred, -1600, 1600)

xgb_valid_metrics = evaluate_regressor(
  y_valid,
  valid_pred,
  "Validation XGBoost ELO-difference regression",
)

plot_regression_diagnostics(
  y_valid,
  valid_pred,
  "Validation XGBoost",
)

In [ ]:
xgb_test_metrics = evaluate_regressor(
  y_test,
  test_pred,
  "Test XGBoost ELO-difference regression",
)

plot_regression_diagnostics(
  y_test,
  test_pred,
  "Test XGBoost",
)

## Inspect ELO-difference regression errors

In [ ]:
pred_df = pd.DataFrame({
  "game_index": model_df.loc[test_mask, "game_index"].reset_index(drop=True),
  "true_rating_diff": np.asarray(y_test, dtype=float),
  "pred_rating_diff": np.asarray(test_pred, dtype=float),
  "rating_diff_bin": diff_bins_test.reset_index(drop=True),
  "own_rating_bin": own_bins_test.reset_index(drop=True),
})

pred_df["residual"] = pred_df["pred_rating_diff"] - pred_df["true_rating_diff"]
pred_df["abs_error"] = pred_df["residual"].abs()

pred_df.head(20)

In [ ]:
error_by_diff_bin = (
  pred_df
  .groupby("rating_diff_bin", observed=True)
  .agg(
    n=("abs_error", "size"),
    mae=("abs_error", "mean"),
    median_ae=("abs_error", "median"),
    rmse=("residual", lambda x: np.sqrt(np.mean(np.square(x)))),
    bias=("residual", "mean"),
  )
)

display(error_by_diff_bin)

error_by_own_rating_bin = (
  pred_df
  .groupby("own_rating_bin", observed=True)
  .agg(
    n=("abs_error", "size"),
    mae=("abs_error", "mean"),
    median_ae=("abs_error", "median"),
    rmse=("residual", lambda x: np.sqrt(np.mean(np.square(x)))),
    bias=("residual", "mean"),
  )
)

display(error_by_own_rating_bin)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
pred_df.boxplot(column="abs_error", by="rating_diff_bin", ax=ax)
ax.set_xlabel("True ELO-difference bin")
ax.set_ylabel("Absolute error [ELO]")
ax.set_title("Absolute error by ELO-difference bin")
fig.suptitle("")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(pred_df["true_rating_diff"], pred_df["abs_error"], alpha=0.35, s=18)
ax.set_xlabel("True ELO difference")
ax.set_ylabel("Absolute error [ELO]")
ax.set_title("Absolute error vs true ELO difference")
plt.tight_layout()
plt.show()

## Learning curves from XGBoost eval history

In [ ]:
results = xgb_reg.evals_result()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(results["validation_0"]["rmse"], label="Train")
ax.plot(results["validation_1"]["rmse"], label="Valid")
ax.set_xlabel("Boosting round")
ax.set_ylabel("RMSE [ELO difference]")
ax.set_title("XGBoost learning curve")
ax.legend()
plt.tight_layout()
plt.show()

## Feature importance

This checks whether the model is actually using engine-derived features rather
than only metadata such as time control or opening.

In [ ]:
importance_df = pd.DataFrame({
  "feature": X_train.columns,
  "importance": xgb_reg.feature_importances_,
})

importance_df = importance_df.sort_values("importance", ascending=False)
importance_df.head(30)

In [ ]:
top_importance = importance_df.head(25).copy()

fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(
  top_importance["feature"][::-1],
  top_importance["importance"][::-1],
)
ax.set_xlabel("XGBoost feature importance")
ax.set_title("Top features")
plt.tight_layout()
plt.show()

## Save artifacts

In [ ]:
model_path = MODEL_DIR / "xgb_stockfish_elo_difference_regressor.joblib"
features_path = MODEL_DIR / "xgb_stockfish_elo_difference_regressor_features.joblib"
player_table_path = PROCESSED_DIR / "stockfish_player_rows_elo_difference_regression.parquet"
predictions_path = PROCESSED_DIR / "stockfish_xgb_elo_difference_test_predictions.csv"

joblib.dump(xgb_reg, model_path)
joblib.dump(list(X_train.columns), features_path)
df_player.to_parquet(player_table_path, index=False)
pred_df.to_csv(predictions_path, index=False)

print(f"Wrote: {model_path}")
print(f"Wrote: {features_path}")
print(f"Wrote: {player_table_path}")
print(f"Wrote: {predictions_path}")

## Suggested next experiments

1. Compare this ELO-difference model against the absolute-ELO Stockfish model
   using MAE/RMSE and residual plots.
2. Compare against the raw-move XGBoost regression using the same sampled games.
3. Run an ablation without `result`, `own_result_score`, and `opp_result_score`
   to test whether post-game result information is doing too much work.
4. Increase `N_GAMES_ENGINE` to 2,000-5,000 after the pipeline works locally.
5. Compare `ENGINE_DEPTH=6`, `8`, and `10` on the same game sample.
6. Try only opening plies versus all analyzed plies to test whether early move
   quality already contains most of the rating-difference signal.
